```markdown
1. Ortam Değişkenlerini Yükleme
    os: İşletim sistemi ile etkileşim için

    dotenv: .env dosyasındaki gizli bilgileri yüklemek için

    load_dotenv(): .env dosyasını yükler

    os.getenv(): Ortam değişkenlerini okur

    Amacı: Gizli bilgileri (şifreler, API key'ler) güvenli şekilde saklamak
```

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

NEO4J_URI =os.getenv("NEO4J_URI")
NEO4J_USERNAME =os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD =os.getenv("NEO4J_PASSWORD")


```markdown
2. LLM (Language Model) Kurulumu
    ChatGroq: Groq API'sini kullanmak için

    model_name="Gemma2-9b-It": Google'ın Gemma2 modelinin 9B parametreli versiyonu

    Amacı: Doğal dil işleme ve Cypher sorgu oluşturma için AI modeli
```

In [2]:
from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="Gemma2-9b-It"
)

```markdown
3. Neo4j Graph Database Bağlantısı
    Neo4jGraph: Neo4j veritabanına bağlanmak için

    refresh_schema=False: Schema'yı otomatik yenileme (performans için)

    Amacı: Graph veritabanına bağlantı kurmak
```

In [3]:
from langchain_community.graphs import Neo4jGraph

#NEO4J_URI = "neo4j+s://0df55adf.databases.neo4j.io"
#NEO4J_USERNAME = "neo4j"
#NEO4J_PASSWORD = "rIbt9q_Z95vAY3-tA9srqbcHRKRAiFhgLEpr2IkKLqw"

# refresh_schema=False ile başlat
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    refresh_schema=False  # Bu önemli!
)


C:\Users\murat\AppData\Local\Temp\ipykernel_28716\45294948.py:8: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


```markdown
4. Veri Yükleme Sorgusu
    LOAD CSV: CSV dosyasını yükler

    MERGE: Node varsa kullanır, yoksa oluşturur

    FOREACH: Dizi elemanları için döngü

    split(row.director,'|'): "|" ile ayrılmış stringleri parçalar

    Yapılan işlem:

        Film node'ları oluşturur

        Yönetmenleri bağlar (DIRECTED_BY)

        Aktörleri bağlar (ACTED_IN)

        Türleri bağlar (HAS_GENRE)
```

In [4]:
# Dataset Moview
moview_query = """
LOAD CSV WITH HEADERS FROM 
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE (m:movie{id: row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)

FOREACH (director in split(row.director,'|') | 
    MERGE (p:Person {name: trim(director)})
    MERGE (p)-[:DIRECTED_BY]->(m)
)
FOREACH (actor in split(row.actors,'|') | 
    MERGE (p:Person {name: trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m)
)
FOREACH (genre in split(row.genres,'|') |
    MERGE (g:Genre {name: trim(genre)})
    MERGE (m)-[:HAS_GENRE]->(g)
)
"""
moview_query

"\nLOAD CSV WITH HEADERS FROM \n'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row\n\nMERGE (m:movie{id: row.movieId})\nSET m.released = date(row.released),\n    m.title = row.title,\n    m.imdbRating = toFloat(row.imdbRating)\n\nFOREACH (director in split(row.director,'|') | \n    MERGE (p:Person {name: trim(director)})\n    MERGE (p)-[:DIRECTED_BY]->(m)\n)\nFOREACH (actor in split(row.actors,'|') | \n    MERGE (p:Person {name: trim(actor)})\n    MERGE (p)-[:ACTED_IN]->(m)\n)\nFOREACH (genre in split(row.genres,'|') |\n    MERGE (g:Genre {name: trim(genre)})\n    MERGE (m)-[:HAS_GENRE]->(g)\n)\n"

```markdown
5. Schema Güncelleme ve Görüntüleme
    graph.query(): Cypher sorgusunu çalıştırır

    graph.refresh_schema(): Database schema'sını yeniler

    print(graph.schema): Schema'yı gösterir

    Çıktı: Node property'leri ve relationship'ları listeler
```

In [5]:
graph.query(moview_query)
graph.refresh_schema()
print(graph.schema)

Node properties:
CEO {name: STRING, POB: STRING, YOB: INTEGER}
Employee {name: STRING, POB: STRING, YOB: INTEGER}
Company {name: STRING}
Country {name: STRING}
Person {name: STRING, born: INTEGER}
Movie {title: STRING, released: INTEGER}
movie {id: STRING, title: STRING, released: DATE, imdbRating: FLOAT}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Person)-[:ACTED_IN]->(:movie)
(:Person)-[:DIRECTED_BY]->(:movie)
(:movie)-[:HAS_GENRE]->(:Genre)


In [6]:
from langchain.chains import GraphCypherQAChain
chain = GraphCypherQAChain.from_llm(llm,graph=graph, verbose=True,allow_dangerous_requests=True)
chain

GraphCypherQAChain(verbose=True, graph=<langchain_community.graphs.neo4j_graph.Neo4jGraph object at 0x000002754D269DF0>, cypher_generation_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nThe question is:\n{question}'), llm=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000002754F6CE7B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002754FCBCC80>, model_name=

```markdown
6. Few-Shot Prompt Template Oluşturma

    Few-shot learning: LLM'e örneklerle öğretme

    examples: Soru-cevap çiftleri

    {{title: 'Casino'}}: Çift süslü parantez (formatting için)

    Yapısı:

        prefix: LLM'in rol tanımı

        examples: Öğrenme örnekleri

        suffix: Kullanıcı sorusu için template

        input_variables: Değişkenler
```

In [7]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# Düzeltilmiş örnekler - çift süslü parantez kullanın
examples = [{
    "question": "Who is the director of movie Casino?",
    "query": "MATCH (m:movie {{title: 'Casino'}})-[:DIRECTED_BY]->(p:Person) RETURN p.name AS director"
}, {
    "question": "What are the genres of the movie Inception?",
    "query": "MATCH (m:movie {{title: 'Inception'}})-[:HAS_GENRE]->(g:Genre) RETURN g.name AS genre"
}, {
    "question": "List all movies released in 1994.",
    "query": "MATCH (m:movie) WHERE m.released.year = 1994 RETURN m.title AS title"
}, {
    "question": "Who acted in the movie The Matrix?",
    "query": "MATCH (m:movie {{title: 'The Matrix'}})<-[:ACTED_IN]-(p:Person) RETURN p.name AS actor"
}, {
    "question": "What is the IMDb rating of the movie Titanic?",
    "query": "MATCH (m:movie {{title: 'Titanic'}}) RETURN m.imdbRating AS rating"
}]

example_prompt = PromptTemplate.from_template(
    "User Input: {question}\n"
    "Cypher Query: {query}\n"
)

prompt = FewShotPromptTemplate(
    examples=examples[:5],
    example_prompt=example_prompt,
    prefix="You are a Neo4j expert. Given an input question, create a very accurate Cypher query to answer it.\n",
    suffix="User input: {question}\nCypher Query: ",
    input_variables=["question"]
)



```markdown
7. GraphCypherQAChain Oluşturma
    GraphCypherQAChain: Soru-cevap zinciri

    llm=llm: Kullanılacak AI modeli

    graph=graph: Bağlı graph database

    cypher_prompt=prompt: Özel prompt template

    verbose=True: Detaylı log gösterimi

    allow_dangerous_requests=True: Güvenlik kısıtlamalarını kaldırır
```

In [15]:
llm
chain = GraphCypherQAChain.from_llm(llm, graph=graph, cypher_prompt=prompt, verbose=True, allow_dangerous_requests=True)

```markdown
8. Sorgu Çalıştırma
    Soru alınır: "Which actors played in the movie Toy Story?"

    LLM prompt'u işler: Few-shot örnekleri kullanarak

    Cypher oluşturur: MATCH (m:movie {title: 'Toy Story'})<-[:ACTED_IN]-(p:Person) RETURN p.name AS actor

    Sorgu çalıştırılır: Neo4j'de query çalıştırılır

    Sonuç formatlanır: LLM sonucu insan diline çevirir

    Cevap döndürülür: "Tom Hanks, Jim Varney, Tim Allen, and Don Rickles"
```

In [20]:
chain.invoke("Which actors played in the movie Toy Story?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (m:movie {title: 'Toy Story'})<-[:ACTED_IN]-(p:Person)
RETURN p.name AS actor

Full Context:
[{'actor': 'Tom Hanks'}, {'actor': 'Jim Varney'}, {'actor': 'Tim Allen'}, {'actor': 'Don Rickles'}]

> Finished chain.


{'query': 'Which actors played in the movie Toy Story?',
 'result': 'Tom Hanks, Jim Varney, Tim Allen, and Don Rickles played in the movie Toy Story.  \n'}